###  DFE Implementation and compatible with our datasets

In [14]:
# ==================== CELL 1: آموزش مدل پایه DFE روی ISCX_TOR ====================
# این سلول را فقط یک بار برای آموزش مدل اجرا کنید

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import os
import math
import time
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==================== کد اصلی DFE (بدون تغییر) ====================

class DFEBackbone(nn.Module):
    def __init__(self, input_channels=1, embedding_dim=256, input_features=81):
        super(DFEBackbone, self).__init__()
        self.input_features = input_features
        self.input_size = self._calculate_input_size(input_features)
        
        self.conv1 = nn.Conv2d(input_channels, 64, kernel_size=3, padding=1, stride=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=2)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=1)
        self.bn4 = nn.BatchNorm2d(64)
        
        self.conv5 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=1)
        self.bn5 = nn.BatchNorm2d(64)
        
        self.conv6 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.bn6 = nn.BatchNorm2d(128)
        self.conv7 = nn.Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn7 = nn.BatchNorm2d(128)
        
        self.conv8 = nn.Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn8 = nn.BatchNorm2d(128)
        self.conv9 = nn.Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn9 = nn.BatchNorm2d(128)
        
        self.conv10 = nn.Conv2d(128, 256, kernel_size=3, padding=1, stride=2)
        self.bn10 = nn.BatchNorm2d(256)
        self.conv11 = nn.Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn11 = nn.BatchNorm2d(256)
        self.conv12 = nn.Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn12 = nn.BatchNorm2d(256)
        self.conv13 = nn.Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn13 = nn.BatchNorm2d(256)
        
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(256, 256)
        self.fc2 = nn.Linear(256, embedding_dim)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.5)
        
        self.res1 = nn.Sequential(nn.Conv2d(input_channels, 64, kernel_size=1, stride=1), nn.BatchNorm2d(64))
        self.res2 = nn.Sequential(nn.Conv2d(64, 64, kernel_size=1, stride=2), nn.BatchNorm2d(64))
        self.res4 = nn.Sequential(nn.Conv2d(64, 128, kernel_size=1, stride=2), nn.BatchNorm2d(128))
        self.res6 = nn.Sequential(nn.Conv2d(128, 256, kernel_size=1, stride=2), nn.BatchNorm2d(256))
        
        self._initialize_weights()
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def _calculate_input_size(self, num_features):
        sqrt = math.isqrt(num_features)
        if sqrt * sqrt == num_features:
            return sqrt
        return sqrt + 1
    
    def forward(self, x, return_features=False):
        features = []
        
        x1 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.relu(self.bn2(self.conv2(x1)))
        x1 = x1 + self.res1(x)
        features.append(x1)
        
        x2 = self.relu(self.bn3(self.conv3(x1)))
        x2 = self.relu(self.bn4(self.conv4(x2)))
        x2 = x2 + self.res2(x1)
        features.append(x2)
        
        x3 = self.relu(self.bn5(self.conv5(x2)))
        x3 = x3 + x2
        features.append(x3)
        
        x4 = self.relu(self.bn6(self.conv6(x3)))
        x4 = self.relu(self.bn7(self.conv7(x4)))
        x4 = x4 + self.res4(x2)
        features.append(x4)
        
        x5 = self.relu(self.bn8(self.conv8(x4)))
        x5 = self.relu(self.bn9(self.conv9(x5)))
        x5 = x5 + x4
        features.append(x5)
        
        x6 = self.relu(self.bn10(self.conv10(x5)))
        x6 = self.relu(self.bn11(self.conv11(x6)))
        x6 = self.relu(self.bn12(self.conv12(x6)))
        x6 = self.relu(self.bn13(self.conv13(x6)))
        x6 = x6 + self.res6(x4)
        features.append(x6)
        
        x_pool = self.adaptive_pool(x6)
        x_pool = x_pool.view(x_pool.size(0), -1)
        embedding = self.relu(self.fc1(x_pool))
        embedding = self.dropout(embedding)
        embedding = self.fc2(embedding)
        
        if return_features:
            return embedding, features
        return embedding


class DFEComplete(nn.Module):
    def __init__(self, input_channels=1, embedding_dim=256, input_features=81):
        super(DFEComplete, self).__init__()
        self.backbone = DFEBackbone(input_channels, embedding_dim, input_features)
        
    def forward(self, x, return_features=False):
        if return_features:
            return self.backbone(x, return_features=True)
        return self.backbone(x)


class FeatureCompressor(nn.Module):
    def __init__(self, beta_values=[0.01, 0.02, 0.03, 0.04, 0.05, 0.05]):
        super(FeatureCompressor, self).__init__()
        self.register_buffer('beta', torch.tensor(beta_values, dtype=torch.float32))
        
    def compute_compression_loss(self, features):
        total_loss = torch.tensor(0.0, device=features[0].device)
        
        for i in range(len(features) - 1):
            f_i = features[i]
            f_j = features[i + 1]
            
            batch_size = min(f_i.size(0), f_j.size(0))
            if batch_size < 2:
                continue
                
            f_i = f_i[:batch_size]
            f_j = f_j[:batch_size]
            
            f_i_flat = f_i.view(f_i.size(0), -1)
            f_j_flat = f_j.view(f_j.size(0), -1)
            
            f_i_norm = F.normalize(f_i_flat, p=2, dim=1)
            f_j_norm = F.normalize(f_j_flat, p=2, dim=1)
            
            var_i = torch.var(f_i_norm)
            var_j = torch.var(f_j_norm)
            
            eps = 1e-6
            log_var_i = torch.log(var_i + eps)
            log_var_j = torch.log(var_j + eps)
            
            loss_i = (log_var_i - log_var_j) ** 2
            
            if i < len(self.beta):
                total_loss = total_loss + self.beta[i] * loss_i
                
        return total_loss


class ModifiedTripletLoss(nn.Module):
    def __init__(self, alpha1=0.5, alpha2=2.0, eta=0.5):
        super(ModifiedTripletLoss, self).__init__()
        self.alpha1 = alpha1
        self.alpha2 = alpha2
        self.eta = eta
        
    def forward(self, anchor, positive, negative):
        d_pos = F.pairwise_distance(anchor, positive, p=2)
        d_neg = F.pairwise_distance(anchor, negative, p=2)
        
        inter_class_loss = F.relu(d_pos - d_neg + self.alpha1)
        intra_class_loss = F.relu(d_pos - self.alpha2)
        
        total_loss = inter_class_loss.mean() + self.eta * intra_class_loss.mean()
        return total_loss


class TripletDataset:
    def __init__(self, X, y, samples_per_class=100, input_size=None):
        self.X = X
        self.y = y
        self.input_size = input_size
        self.triplets = self._generate_triplets(X, y, samples_per_class)
        
    def _generate_triplets(self, X, y, samples_per_class):
        triplets = []
        unique_classes = np.unique(y)
        class_indices = {cls: np.where(y == cls)[0] for cls in unique_classes}
        
        for cls in unique_classes:
            cls_indices = class_indices[cls]
            other_classes = [c for c in unique_classes if c != cls]
            
            if len(cls_indices) < 2 or len(other_classes) == 0:
                continue
                
            n_triplets = min(samples_per_class, len(cls_indices))
            
            for _ in range(n_triplets):
                anchor_idx = np.random.choice(cls_indices)
                anchor = X[anchor_idx]
                
                pos_candidates = [idx for idx in cls_indices if idx != anchor_idx]
                if not pos_candidates:
                    continue
                pos_idx = np.random.choice(pos_candidates)
                positive = X[pos_idx]
                
                neg_cls = np.random.choice(other_classes)
                neg_indices = class_indices[neg_cls]
                neg_idx = np.random.choice(neg_indices)
                negative = X[neg_idx]
                
                triplets.append([anchor, positive, negative])
                
        return np.array(triplets)
    
    def __len__(self):
        return len(self.triplets)
    
    def _pad_to_square(self, flat_vector, target_size):
        total_elements = target_size * target_size
        current_len = len(flat_vector)
        
        if current_len == total_elements:
            return flat_vector.reshape(1, target_size, target_size)
        elif current_len < total_elements:
            padded = np.pad(flat_vector, (0, total_elements - current_len), 'constant', constant_values=0)
            return padded.reshape(1, target_size, target_size)
        else:
            cut = flat_vector[:total_elements]
            return cut.reshape(1, target_size, target_size)

    def __getitem__(self, idx):
        triplet = self.triplets[idx]
        
        anchor = self._pad_to_square(triplet[0], self.input_size)
        positive = self._pad_to_square(triplet[1], self.input_size)
        negative = self._pad_to_square(triplet[2], self.input_size)
        
        return torch.FloatTensor(anchor), torch.FloatTensor(positive), torch.FloatTensor(negative)


class DFE:
    def __init__(self, embedding_dim=256, alpha1=0.5, alpha2=2.0, eta=0.5,
                 batch_size=32, learning_rate=0.001, epochs=200, lambda_weight=1.0,
                 beta_values=None, k_neighbors=5):
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        self.embedding_dim = embedding_dim
        self.alpha1 = alpha1
        self.alpha2 = alpha2
        self.eta = eta
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.lambda_weight = lambda_weight
        self.beta_values = beta_values or [0.01, 0.02, 0.03, 0.04, 0.05, 0.05]
        self.k_neighbors = k_neighbors
        
        self.input_features = None
        self.input_size = None
        self.model = None
        self.template_library = []
        self.label_encoder = LabelEncoder()
        self.scaler = StandardScaler()
        
    def prepare_data(self, X, y, is_training=True):
        self.input_features = X.shape[1]
        self.input_size = self._calculate_input_size(self.input_features)
        
        if is_training:
            y_encoded = self.label_encoder.fit_transform(y)
            X_scaled = self.scaler.fit_transform(X)
        else:
            y_encoded = self.label_encoder.transform(y)
            X_scaled = self.scaler.transform(X)
        
        return X_scaled, y_encoded
    
    def _calculate_input_size(self, num_features):
        sqrt = math.isqrt(num_features)
        if sqrt * sqrt == num_features:
            return sqrt
        return sqrt + 1
    
    def train(self, X_train, y_train, samples_per_class=1000, verbose=True):
        print(f"\n{'='*60}")
        print("STARTING DFE TRAINING ON Dataset")
        print(f"{'='*60}")
        
        X_train_prep, y_train_encoded = self.prepare_data(X_train, y_train, is_training=True)
        
        print(f"Training samples: {len(X_train_prep)}")
        print(f"Input features: {self.input_features} → Square size: {self.input_size}x{self.input_size}")
        print(f"Number of classes: {len(np.unique(y_train_encoded))}")
        
        triplet_dataset = TripletDataset(
            X_train_prep, y_train_encoded, 
            samples_per_class=samples_per_class, 
            input_size=self.input_size
        )
        triplet_loader = DataLoader(
            triplet_dataset, 
            batch_size=self.batch_size, 
            shuffle=True, 
            drop_last=True, 
            num_workers=0
        )
        
        print(f"Created {len(triplet_dataset)} triplets")
        
        self.model = DFEComplete(
            input_channels=1, 
            embedding_dim=self.embedding_dim, 
            input_features=self.input_features
        ).to(self.device)
        
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"Total parameters: {total_params:,}")
        
        triplet_loss_fn = ModifiedTripletLoss(alpha1=self.alpha1, alpha2=self.alpha2, eta=self.eta)
        feature_compressor = FeatureCompressor(beta_values=self.beta_values).to(self.device)
        
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.epochs, eta_min=1e-6)
        
        best_loss = float('inf')
        patience_counter = 0
        patience = 25
        start_time = time.time()
        
        for epoch in range(self.epochs):
            self.model.train()
            epoch_loss = 0
            epoch_triplet_loss = 0
            epoch_compression_loss = 0
            num_batches = 0
            
            for anchors, positives, negatives in triplet_loader:
                anchors = anchors.to(self.device)
                positives = positives.to(self.device)
                negatives = negatives.to(self.device)
                
                optimizer.zero_grad()
                
                anchor_emb, anchor_features = self.model(anchors, return_features=True)
                positive_emb = self.model(positives)
                negative_emb = self.model(negatives)
                
                triplet_loss = triplet_loss_fn(anchor_emb, positive_emb, negative_emb)
                compression_loss = feature_compressor.compute_compression_loss(anchor_features)
                total_loss = compression_loss + self.lambda_weight * triplet_loss
                
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()
                
                epoch_loss += total_loss.item()
                epoch_triplet_loss += triplet_loss.item()
                epoch_compression_loss += compression_loss.item()
                num_batches += 1
            
            scheduler.step()
            
            if num_batches > 0:
                avg_loss = epoch_loss / num_batches
                
                if verbose and (epoch + 1) % 10 == 0:
                    elapsed = time.time() - start_time
                    avg_triplet = epoch_triplet_loss / num_batches
                    avg_compression = epoch_compression_loss / num_batches
                    print(f"Epoch [{epoch+1:3d}/{self.epochs}] Loss: {avg_loss:.4f} "
                          f"(Triplet: {avg_triplet:.4f}, Comp: {avg_compression:.4f}) Time: {elapsed:.1f}s")
                
                if avg_loss < best_loss:
                    best_loss = avg_loss
                    patience_counter = 0
                else:
                    patience_counter += 1
                    
                if patience_counter >= patience and epoch > 50:
                    print(f"Early stopping at epoch {epoch+1}")
                    break
        
        train_time = time.time() - start_time
        print(f"\n✅ Training completed in {train_time:.2f}s")
        print(f"Best loss: {best_loss:.4f}")
        
        self._build_template_library(X_train_prep, y_train_encoded)
        
    def _build_template_library(self, X, y, samples_per_class=50):
        print(f"\nBuilding template library with {samples_per_class} samples per class...")
        self.model.eval()
        
        total_elements = self.input_size * self.input_size
        unique_classes = np.unique(y)
        
        for cls in unique_classes:
            class_indices = np.where(y == cls)[0]
            n_samples = min(samples_per_class, len(class_indices))
            selected_indices = np.random.choice(class_indices, n_samples, replace=False)
            
            class_samples = X[selected_indices]
            
            padded_samples = []
            for sample in class_samples:
                if len(sample) == total_elements:
                    padded_samples.append(sample.reshape(1, self.input_size, self.input_size))
                elif len(sample) < total_elements:
                    padded = np.pad(sample, (0, total_elements - len(sample)), 'constant', constant_values=0)
                    padded_samples.append(padded.reshape(1, self.input_size, self.input_size))
                else:
                    cut = sample[:total_elements]
                    padded_samples.append(cut.reshape(1, self.input_size, self.input_size))
                    
            class_samples_reshaped = np.array(padded_samples)
            class_samples_tensor = torch.FloatTensor(class_samples_reshaped).to(self.device)
            
            with torch.no_grad():
                embeddings = self.model(class_samples_tensor).cpu().numpy()
            
            for idx, emb in enumerate(embeddings):
                self.template_library.append({'emb': emb, 'label': int(cls)})
        
        print(f"Template library built with {len(self.template_library)} templates")

    def evaluate(self, X_test, y_test, verbose=True):
        X_test_prep, y_test_encoded = self.prepare_data(X_test, y_test, is_training=False)
        y_pred = self._predict_with_template_matching(X_test_prep)
        
        accuracy = accuracy_score(y_test_encoded, y_pred)
        
        if verbose:
            print(f"\n📊 Evaluation Results:")
            print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
        
        return {'accuracy': accuracy}
    
    def predict_batch(self, X):
        X_scaled = self.scaler.transform(X)
        return self._predict_with_template_matching(X_scaled)
    
    def _predict_with_template_matching(self, X):
        self.model.eval()
        predictions = []
        batch_size = 32
        total_elements = self.input_size * self.input_size
        
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                batch_X = X[i:i + batch_size]
                
                padded_batch = []
                for sample in batch_X:
                    if len(sample) == total_elements:
                        padded_batch.append(sample.reshape(1, self.input_size, self.input_size))
                    elif len(sample) < total_elements:
                        padded = np.pad(sample, (0, total_elements - len(sample)), 'constant', constant_values=0)
                        padded_batch.append(padded.reshape(1, self.input_size, self.input_size))
                    else:
                        cut = sample[:total_elements]
                        padded_batch.append(cut.reshape(1, self.input_size, self.input_size))
                        
                batch_X_reshaped = np.array(padded_batch)
                X_tensor = torch.FloatTensor(batch_X_reshaped).to(self.device)
                embeddings = self.model(X_tensor).cpu().numpy()
                
                for emb in embeddings:
                    templates = np.array([t['emb'] for t in self.template_library])
                    labels = np.array([t['label'] for t in self.template_library])
                    
                    diff = templates - emb
                    dists = np.sqrt(np.sum(diff**2, axis=1))
                    
                    k = min(self.k_neighbors, len(dists))
                    k_indices = np.argpartition(dists, k)[:k]
                    k_labels = labels[k_indices]
                    
                    unique_labels, counts = np.unique(k_labels, return_counts=True)
                    majority_label = unique_labels[np.argmax(counts)]
                    predictions.append(majority_label)
        
        return np.array(predictions)
    
    def predict_one(self, x):
        if isinstance(x, dict):
            x = np.array([x[i] for i in range(len(x))])
        x = x.reshape(1, -1)
        return self.predict_batch(x)[0]
    
    def save_model(self, path='Models/DFE_ISCXTOR_base.pth'):
        os.makedirs('Models', exist_ok=True)
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'template_library': self.template_library,
            'label_encoder_classes': self.label_encoder.classes_,
            'scaler_mean': self.scaler.mean_,
            'scaler_scale': self.scaler.scale_,
            'input_features': self.input_features,
            'input_size': self.input_size,
            'parameters': {
                'embedding_dim': self.embedding_dim,
                'alpha1': self.alpha1,
                'alpha2': self.alpha2,
                'eta': self.eta,
                'beta_values': self.beta_values,
                'lambda_weight': self.lambda_weight,
                'k_neighbors': self.k_neighbors
            }
        }, path)
        print(f"✅ Model saved to {path}")
        
    def load_model(self, path='Models/DFE_ISCXTOR_base.pth'):
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        
        self.input_features = checkpoint['input_features']
        self.input_size = checkpoint['input_size']
        
        self.model = DFEComplete(
            input_channels=1,
            embedding_dim=checkpoint['parameters']['embedding_dim'],
            input_features=self.input_features
        ).to(self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        self.template_library = checkpoint['template_library']
        self.label_encoder.classes_ = checkpoint['label_encoder_classes']
        self.scaler.mean_ = checkpoint['scaler_mean']
        self.scaler.scale_ = checkpoint['scaler_scale']
        self.k_neighbors = checkpoint['parameters']['k_neighbors']
        
        print(f"✅ Model loaded from {path}")
        print(f"   Template library size: {len(self.template_library)}")


# ==================== بارگذاری داده  ====================

def load_data(dataset_path='Dataset', csv_filename='UNSW_IoT_original.csv'):

    print(f"\n📂 Loading data from {dataset_path}/{csv_filename}")
    
    df = pd.read_csv(f'{dataset_path}/{csv_filename}')
    print(f"  Total samples: {len(df)}")
    print(f"  Features: {df.shape[1] - 1}")
    print(f"  Classes: {df.iloc[:, -1].nunique()}")
    
    # تقسیم داده (60% train, 20% val, 20% test)
    train_df, temp_df = train_test_split(df, test_size=0.4, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    
    print(f"  Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    
    X_train = train_df.iloc[:, :-1].values.astype(np.float32)
    y_train = train_df.iloc[:, -1].values
    
    X_val = val_df.iloc[:, :-1].values.astype(np.float32)
    y_val = val_df.iloc[:, -1].values
    
    X_test = test_df.iloc[:, :-1].values.astype(np.float32)
    y_test = test_df.iloc[:, -1].values
    
    return X_train, y_train, X_val, y_val, X_test, y_test


# ==================== آموزش مدل پایه ====================

def train_base_model():
    """آموزش مدل پایه DFE روی IUST_MSCN_original و ذخیره آن"""
    
    print("="*80)
    print("TRAINING DFE BASE MODEL ON DATASET")
    print("="*80)
    
    # بارگذاری داده
    X_train, y_train, X_val, y_val, X_test, y_test = load_data()
    
    # ایجاد مدل DFE
    dfe = DFE(
        embedding_dim=256,
        alpha1=0.5,
        alpha2=2.0,
        eta=0.5,
        batch_size=32,
        learning_rate=0.001,
        epochs=80,
        lambda_weight=1.0,
        k_neighbors=5
    )
    
    # آموزش
    dfe.train(X_train, y_train, samples_per_class=1000, verbose=True)
    
    # ارزیابی روی validation
    print("\n" + "="*60)
    print("VALIDATION RESULTS")
    print("="*60)
    val_results = dfe.evaluate(X_val, y_val, verbose=True)
    
    # ارزیابی روی test
    print("\n" + "="*60)
    print("TEST RESULTS")
    print("="*60)
    test_results = dfe.evaluate(X_test, y_test, verbose=True)
    
    # ذخیره مدل
    dfe.save_model('Models/DFE_base.pth')
    
    print("\n" + "="*60)
    print("✅ BASE MODEL TRAINING COMPLETE")
    print("="*60)
    
    return dfe


# اجرای آموزش
if __name__ == "__main__":
    train_base_model()

Using device: cuda
TRAINING DFE BASE MODEL ON DATASET

📂 Loading data from Dataset/UNSW_IoT_original.csv
  Total samples: 5000
  Features: 18
  Classes: 5
  Train: 3000, Val: 1000, Test: 1000

STARTING DFE TRAINING ON Dataset
Training samples: 3000
Input features: 18 → Square size: 5x5
Number of classes: 5
Created 3000 triplets
Total parameters: 2,912,320
Epoch [ 10/80] Loss: 0.3306 (Triplet: 0.3171, Comp: 0.0136) Time: 26.2s
Epoch [ 20/80] Loss: 0.2255 (Triplet: 0.2140, Comp: 0.0116) Time: 49.7s
Epoch [ 30/80] Loss: 0.1858 (Triplet: 0.1762, Comp: 0.0095) Time: 73.1s
Epoch [ 40/80] Loss: 0.1342 (Triplet: 0.1259, Comp: 0.0082) Time: 96.4s
Epoch [ 50/80] Loss: 0.1120 (Triplet: 0.1050, Comp: 0.0070) Time: 119.8s
Epoch [ 60/80] Loss: 0.0846 (Triplet: 0.0785, Comp: 0.0061) Time: 143.9s
Epoch [ 70/80] Loss: 0.0665 (Triplet: 0.0610, Comp: 0.0055) Time: 170.2s
Epoch [ 80/80] Loss: 0.0657 (Triplet: 0.0603, Comp: 0.0055) Time: 193.9s

✅ Training completed in 193.89s
Best loss: 0.0652

Building t

In [17]:
# ==================== CELL 2: تست و ارزیابی مدل DFE روی  ====================
# این سلول را بعد از آموزش مدل پایه اجرا کنید

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import os
import math
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==================== تعریف کلاس‌های DFE (برای بارگذاری مدل) ====================

class DFEBackbone(nn.Module):
    def __init__(self, input_channels=1, embedding_dim=256, input_features=81):
        super(DFEBackbone, self).__init__()
        self.input_features = input_features
        self.input_size = self._calculate_input_size(input_features)
        
        self.conv1 = nn.Conv2d(input_channels, 64, kernel_size=3, padding=1, stride=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=2)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=1)
        self.bn4 = nn.BatchNorm2d(64)
        
        self.conv5 = nn.Conv2d(64, 64, kernel_size=3, padding=1, stride=1)
        self.bn5 = nn.BatchNorm2d(64)
        
        self.conv6 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.bn6 = nn.BatchNorm2d(128)
        self.conv7 = nn.Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn7 = nn.BatchNorm2d(128)
        
        self.conv8 = nn.Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn8 = nn.BatchNorm2d(128)
        self.conv9 = nn.Conv2d(128, 128, kernel_size=3, padding=1, stride=1)
        self.bn9 = nn.BatchNorm2d(128)
        
        self.conv10 = nn.Conv2d(128, 256, kernel_size=3, padding=1, stride=2)
        self.bn10 = nn.BatchNorm2d(256)
        self.conv11 = nn.Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn11 = nn.BatchNorm2d(256)
        self.conv12 = nn.Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn12 = nn.BatchNorm2d(256)
        self.conv13 = nn.Conv2d(256, 256, kernel_size=3, padding=1, stride=1)
        self.bn13 = nn.BatchNorm2d(256)
        
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(256, 256)
        self.fc2 = nn.Linear(256, embedding_dim)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.5)
        
        self.res1 = nn.Sequential(nn.Conv2d(input_channels, 64, kernel_size=1, stride=1), nn.BatchNorm2d(64))
        self.res2 = nn.Sequential(nn.Conv2d(64, 64, kernel_size=1, stride=2), nn.BatchNorm2d(64))
        self.res4 = nn.Sequential(nn.Conv2d(64, 128, kernel_size=1, stride=2), nn.BatchNorm2d(128))
        self.res6 = nn.Sequential(nn.Conv2d(128, 256, kernel_size=1, stride=2), nn.BatchNorm2d(256))
        
        self._initialize_weights()
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def _calculate_input_size(self, num_features):
        sqrt = math.isqrt(num_features)
        if sqrt * sqrt == num_features:
            return sqrt
        return sqrt + 1
    
    def forward(self, x, return_features=False):
        features = []
        
        x1 = self.relu(self.bn1(self.conv1(x)))
        x1 = self.relu(self.bn2(self.conv2(x1)))
        x1 = x1 + self.res1(x)
        features.append(x1)
        
        x2 = self.relu(self.bn3(self.conv3(x1)))
        x2 = self.relu(self.bn4(self.conv4(x2)))
        x2 = x2 + self.res2(x1)
        features.append(x2)
        
        x3 = self.relu(self.bn5(self.conv5(x2)))
        x3 = x3 + x2
        features.append(x3)
        
        x4 = self.relu(self.bn6(self.conv6(x3)))
        x4 = self.relu(self.bn7(self.conv7(x4)))
        x4 = x4 + self.res4(x2)
        features.append(x4)
        
        x5 = self.relu(self.bn8(self.conv8(x4)))
        x5 = self.relu(self.bn9(self.conv9(x5)))
        x5 = x5 + x4
        features.append(x5)
        
        x6 = self.relu(self.bn10(self.conv10(x5)))
        x6 = self.relu(self.bn11(self.conv11(x6)))
        x6 = self.relu(self.bn12(self.conv12(x6)))
        x6 = self.relu(self.bn13(self.conv13(x6)))
        x6 = x6 + self.res6(x4)
        features.append(x6)
        
        x_pool = self.adaptive_pool(x6)
        x_pool = x_pool.view(x_pool.size(0), -1)
        embedding = self.relu(self.fc1(x_pool))
        embedding = self.dropout(embedding)
        embedding = self.fc2(embedding)
        
        if return_features:
            return embedding, features
        return embedding


class DFEComplete(nn.Module):
    def __init__(self, input_channels=1, embedding_dim=256, input_features=81):
        super(DFEComplete, self).__init__()
        self.backbone = DFEBackbone(input_channels, embedding_dim, input_features)
        
    def forward(self, x, return_features=False):
        if return_features:
            return self.backbone(x, return_features=True)
        return self.backbone(x)


class DFE:
    def __init__(self, k_neighbors=5):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.k_neighbors = k_neighbors
        self.input_features = None
        self.input_size = None
        self.model = None
        self.template_library = []
        self.label_encoder = LabelEncoder()
        self.scaler = StandardScaler()
        
    def _calculate_input_size(self, num_features):
        sqrt = math.isqrt(num_features)
        if sqrt * sqrt == num_features:
            return sqrt
        return sqrt + 1
    
    def load_model(self, path='Models/DFE_base.pth'):
        print(f"📂 Loading model from {path}...")
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        
        self.input_features = checkpoint['input_features']
        self.input_size = checkpoint['input_size']
        
        self.model = DFEComplete(
            input_channels=1,
            embedding_dim=checkpoint['parameters']['embedding_dim'],
            input_features=self.input_features
        ).to(self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        self.template_library = checkpoint['template_library']
        self.label_encoder.classes_ = checkpoint['label_encoder_classes']
        self.scaler.mean_ = checkpoint['scaler_mean']
        self.scaler.scale_ = checkpoint['scaler_scale']
        self.k_neighbors = checkpoint['parameters']['k_neighbors']
        
        print(f"✅ Model loaded: {len(self.template_library)} templates, {len(self.label_encoder.classes_)} classes")
    
    def predict_batch(self, X):
        X_scaled = self.scaler.transform(X)
        return self._predict_with_template_matching(X_scaled)
    
    def _predict_with_template_matching(self, X):
        self.model.eval()
        predictions = []
        batch_size = 32
        total_elements = self.input_size * self.input_size
        
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                batch_X = X[i:i + batch_size]
                
                padded_batch = []
                for sample in batch_X:
                    if len(sample) == total_elements:
                        padded_batch.append(sample.reshape(1, self.input_size, self.input_size))
                    elif len(sample) < total_elements:
                        padded = np.pad(sample, (0, total_elements - len(sample)), 'constant', constant_values=0)
                        padded_batch.append(padded.reshape(1, self.input_size, self.input_size))
                    else:
                        cut = sample[:total_elements]
                        padded_batch.append(cut.reshape(1, self.input_size, self.input_size))
                        
                batch_X_reshaped = np.array(padded_batch)
                X_tensor = torch.FloatTensor(batch_X_reshaped).to(self.device)
                embeddings = self.model(X_tensor).cpu().numpy()
                
                for emb in embeddings:
                    templates = np.array([t['emb'] for t in self.template_library])
                    labels = np.array([t['label'] for t in self.template_library])
                    
                    diff = templates - emb
                    dists = np.sqrt(np.sum(diff**2, axis=1))
                    
                    k = min(self.k_neighbors, len(dists))
                    k_indices = np.argpartition(dists, k)[:k]
                    k_labels = labels[k_indices]
                    
                    unique_labels, counts = np.unique(k_labels, return_counts=True)
                    majority_label = unique_labels[np.argmax(counts)]
                    predictions.append(majority_label)
        
        return np.array(predictions)
    
    def predict_one(self, x):
        if isinstance(x, dict):
            x = np.array([x[i] for i in range(len(x))])
        x = x.reshape(1, -1)
        return self.predict_batch(x)[0]
    
    def evaluate(self, X_test, y_test, verbose=True):
        X_test_prep = self.scaler.transform(X_test)
        y_test_encoded = self.label_encoder.transform(y_test)
        y_pred = self._predict_with_template_matching(X_test_prep)
        
        accuracy = accuracy_score(y_test_encoded, y_pred)
        precision = precision_score(y_test_encoded, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test_encoded, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test_encoded, y_pred, average='weighted', zero_division=0)
        
        if verbose:
            print(f"\n📊 Evaluation Results:")
            print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
            print(f"  Precision: {precision:.4f}")
            print(f"  Recall:    {recall:.4f}")
            print(f"  F1-Score:  {f1:.4f}")
        
        return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1_score': f1}


# ==================== توابع بارگذاری داده ====================

def load_test_data(dataset_path='Dataset', csv_filename='UNSW_IoT_original.csv'):

    df = pd.read_csv(f'{dataset_path}/{csv_filename}')
    
    # تقسیم داده (60% train, 20% val, 20% test) - مشابه سلول اول
    train_df, temp_df = train_test_split(df, test_size=0.4, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    
    X_test = test_df.iloc[:, :-1].values.astype(np.float32)
    y_test = test_df.iloc[:, -1].values
    
    return X_test, y_test


def load_perturbation_data(attack_type='FGSM', test_perturb_levels=None, scaler=None, label_encoder=None, base_path='Dataset'):
    """بارگذاری داده‌های perturbed"""
    if test_perturb_levels is None:
        test_perturb_levels = [0.1, 0.5, 1.0, 2.0, 5.0]
    
    datasets = []
    
    for perturb_level in sorted(test_perturb_levels):
        file_path = f'{base_path}/{attack_type}/{attack_type}_eps_{perturb_level}.csv'
        
        if not os.path.exists(file_path):
            file_path = f'{base_path}/{attack_type}/{attack_type}_eps_{perturb_level}.csv'
        
        if not os.path.exists(file_path):
            print(f"⚠️ Warning: {file_path} not found, skipping...")
            continue
            
        df = pd.read_csv(file_path)
        X = df.iloc[:, :-1].values.astype(np.float32)
        y = df.iloc[:, -1].values
        
        if scaler is not None:
            X_scaled = scaler.transform(X)
        else:
            X_scaled = X
        
        if label_encoder is not None:
            y_encoded = label_encoder.transform(y)
        else:
            y_encoded = y
        
        datasets.append({
            'X': X_scaled,
            'y': y_encoded,
            'perturb_level': perturb_level,
            'original_y': y
        })
        print(f"  Loaded {attack_type} eps={perturb_level}: {len(X_scaled)} samples")
    
    return datasets


# ==================== توابع ارزیابی ====================

def test_dfe_on_perturbed_data(dfe, X_test, y_test, perturb_level, verbose=True):
    """تست DFE روی داده‌های perturbed"""
    y_pred = dfe.predict_batch(X_test)
    y_true = y_test
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    if verbose:
        print(f"  Level {perturb_level}: Acc={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")
    
    return {
        'Perturbation Level': perturb_level,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    }


def run_evaluation(attack_type='FGSM', test_perturb_levels=None):
    """اجرای کامل ارزیابی DFE روی داده‌های perturbed"""
    
    if test_perturb_levels is None:
        test_perturb_levels = [0.1, 0.5, 1.0, 2.0, 5.0]
    
    print("="*80)
    print("DFE MODEL EVALUATION - PERTURBED DATA")
    print(f"Attack Type: {attack_type}")
    print("="*80)
    
    # بارگذاری مدل آموزش دیده
    model_path = 'Models/DFE_base.pth'
    
    if not os.path.exists(model_path):
        print(f"\n❌ Model not found at {model_path}")
        print("Please run CELL 1 (training) first.")
        return None
    
    dfe = DFE()
    dfe.load_model(model_path)
    
    # بارگذاری داده تست تمیز
    X_test_clean, y_test_clean = load_test_data()
    
    # ارزیابی روی داده تمیز
    print("\n" + "="*60)
    print("CLEAN DATA EVALUATION")
    print("="*60)
    clean_results = dfe.evaluate(X_test_clean, y_test_clean, verbose=True)
    
    # بارگذاری داده‌های perturbed
    print(f"\n📂 Loading perturbed data (attack type: {attack_type})...")
    perturb_datasets = load_perturbation_data(
        attack_type=attack_type,
        test_perturb_levels=test_perturb_levels,
        scaler=dfe.scaler,
        label_encoder=dfe.label_encoder,
        base_path='Dataset'
    )
    
    if len(perturb_datasets) == 0:
        print(f"No perturbed datasets found for attack type: {attack_type}")
        return
    
    # تست روی داده‌های perturbed
    print("\n" + "="*60)
    print("PERTURBED DATA EVALUATION")
    print("="*60)
    
    results = []
    for dataset in perturb_datasets:
        result = test_dfe_on_perturbed_data(
            dfe,
            dataset['X'],
            dataset['y'],
            dataset['perturb_level'],
            verbose=True
        )
        results.append(result)
    
    # ذخیره نتایج
    df_results = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print("FINAL RESULTS SUMMARY")
    print("="*60)
    print(df_results[['Perturbation Level', 'Accuracy', 'Precision', 'Recall', 'F1 Score']].to_string(index=False))
    
    # ذخیره در فایل
    os.makedirs('Results', exist_ok=True)
    output_file = f'Results/dfe_{attack_type}_results.csv'
    df_results.to_csv(output_file, index=False)
    
    # ذخیره نتایج داده تمیز
    clean_df = pd.DataFrame([{
        'Perturbation Level': 'Clean',
        'Accuracy': clean_results['accuracy'],
        'Precision': clean_results['precision'],
        'Recall': clean_results['recall'],
        'F1 Score': clean_results['f1_score']
    }])
    clean_df.to_csv(f'Results/dfe_{attack_type}_clean_results.csv', index=False)
    
    print(f"\n💾 Results saved to:")
    print(f"  - {output_file}")
    print(f"  - Results/dfe_{attack_type}_clean_results.csv")
    
    return df_results, clean_results



# ==================== اجرای اصلی ====================

def main(ATTACK_TYPE,TEST_PERTURB_LEVELS):
    """اجرای اصلی ارزیابی"""
        
    print("="*80)
    print("DFE (Deep Feature Extraction) - Evaluation on Dataset")
    print("Method: Triplet Loss + Feature Compression + Template Matching")
    print("Architecture: ResNet-like CNN with 13 convolutional layers")
    print("")
    print("EVALUATION METRICS: Accuracy, Precision, Recall, F1-Score")
    print("="*80)
    
    # اجرای ارزیابی
    results, clean_results = run_evaluation(ATTACK_TYPE, TEST_PERTURB_LEVELS)
    
    print("\n" + "="*80)
    print("✅ EVALUATION COMPLETE")
    print("="*80)
    print("\n📁 Generated files:")
    print(f"  1. Results/dfe_{ATTACK_TYPE}_results.csv - Perturbed data results")
    print(f"  2. Results/dfe_{ATTACK_TYPE}_clean_results.csv - Clean data results")
    print(f"  3. Results/comparison_dfe_af_{ATTACK_TYPE}.csv - DFE vs AF comparison")
    
    return results, clean_results


Using device: cuda


In [18]:
# اجرای تست
if __name__ == "__main__":
    # تنظیمات
    ATTACK_TYPE = 'FGSM'  # می‌توان به 'PGD' یا 'DeepFool' تغییر داد
    TEST_PERTURB_LEVELS = [0.01, 0.05, 0.1, 0.2, 0.5]
    results, clean_results = main(ATTACK_TYPE,TEST_PERTURB_LEVELS)

DFE (Deep Feature Extraction) - Evaluation on Dataset
Method: Triplet Loss + Feature Compression + Template Matching
Architecture: ResNet-like CNN with 13 convolutional layers

EVALUATION METRICS: Accuracy, Precision, Recall, F1-Score
DFE MODEL EVALUATION - PERTURBED DATA
Attack Type: FGSM
📂 Loading model from Models/DFE_base.pth...
✅ Model loaded: 250 templates, 5 classes

CLEAN DATA EVALUATION

📊 Evaluation Results:
  Accuracy:  0.8990 (89.90%)
  Precision: 0.9000
  Recall:    0.8990
  F1-Score:  0.8983

📂 Loading perturbed data (attack type: FGSM)...
  Loaded FGSM eps=0.01: 1000 samples
  Loaded FGSM eps=0.05: 1000 samples
  Loaded FGSM eps=0.1: 1000 samples
  Loaded FGSM eps=0.2: 1000 samples
  Loaded FGSM eps=0.5: 1000 samples

PERTURBED DATA EVALUATION
  Level 0.01: Acc=0.2190, Precision=0.1089, Recall=0.2190, F1=0.1016
  Level 0.05: Acc=0.2210, Precision=0.1153, Recall=0.2210, F1=0.1046
  Level 0.1: Acc=0.2250, Precision=0.1219, Recall=0.2250, F1=0.1103
  Level 0.2: Acc=0.2270, 

In [19]:
# اجرای تست
if __name__ == "__main__":
    # تنظیمات
    ATTACK_TYPE = 'PGD'  # می‌توان به 'PGD' یا 'DeepFool' تغییر داد
    TEST_PERTURB_LEVELS = [0.01, 0.05, 0.1, 0.2, 0.5]
    results, clean_results = main(ATTACK_TYPE,TEST_PERTURB_LEVELS)

DFE (Deep Feature Extraction) - Evaluation on Dataset
Method: Triplet Loss + Feature Compression + Template Matching
Architecture: ResNet-like CNN with 13 convolutional layers

EVALUATION METRICS: Accuracy, Precision, Recall, F1-Score
DFE MODEL EVALUATION - PERTURBED DATA
Attack Type: PGD
📂 Loading model from Models/DFE_base.pth...
✅ Model loaded: 250 templates, 5 classes

CLEAN DATA EVALUATION

📊 Evaluation Results:
  Accuracy:  0.8990 (89.90%)
  Precision: 0.9000
  Recall:    0.8990
  F1-Score:  0.8983

📂 Loading perturbed data (attack type: PGD)...
  Loaded PGD eps=0.01: 1000 samples
  Loaded PGD eps=0.05: 1000 samples
  Loaded PGD eps=0.1: 1000 samples
  Loaded PGD eps=0.2: 1000 samples
  Loaded PGD eps=0.5: 1000 samples

PERTURBED DATA EVALUATION
  Level 0.01: Acc=0.2170, Precision=0.1055, Recall=0.2170, F1=0.0997
  Level 0.05: Acc=0.2150, Precision=0.1001, Recall=0.2150, F1=0.0967
  Level 0.1: Acc=0.2150, Precision=0.0988, Recall=0.2150, F1=0.0956
  Level 0.2: Acc=0.2080, Precisi

In [20]:
# اجرای تست
if __name__ == "__main__":
    # تنظیمات
    ATTACK_TYPE = 'CW'  # می‌توان به 'PGD' یا 'DeepFool' تغییر داد
    TEST_PERTURB_LEVELS = [0.1, 0.5, 1.0, 2.0, 5.0]
    results, clean_results = main(ATTACK_TYPE,TEST_PERTURB_LEVELS)

DFE (Deep Feature Extraction) - Evaluation on Dataset
Method: Triplet Loss + Feature Compression + Template Matching
Architecture: ResNet-like CNN with 13 convolutional layers

EVALUATION METRICS: Accuracy, Precision, Recall, F1-Score
DFE MODEL EVALUATION - PERTURBED DATA
Attack Type: CW
📂 Loading model from Models/DFE_base.pth...
✅ Model loaded: 250 templates, 5 classes

CLEAN DATA EVALUATION

📊 Evaluation Results:
  Accuracy:  0.8990 (89.90%)
  Precision: 0.9000
  Recall:    0.8990
  F1-Score:  0.8983

📂 Loading perturbed data (attack type: CW)...
  Loaded CW eps=0.1: 1000 samples
  Loaded CW eps=0.5: 1000 samples
  Loaded CW eps=1.0: 1000 samples
  Loaded CW eps=2.0: 1000 samples
  Loaded CW eps=5.0: 1000 samples

PERTURBED DATA EVALUATION
  Level 0.1: Acc=0.2170, Precision=0.1055, Recall=0.2170, F1=0.0997
  Level 0.5: Acc=0.2170, Precision=0.1046, Recall=0.2170, F1=0.0997
  Level 1.0: Acc=0.2170, Precision=0.1086, Recall=0.2170, F1=0.0999
  Level 2.0: Acc=0.2170, Precision=0.1132, 

In [21]:
# اجرای تست
if __name__ == "__main__":
    # تنظیمات
    ATTACK_TYPE = 'DeepFool'  # می‌توان به 'PGD' یا 'DeepFool' تغییر داد
    TEST_PERTURB_LEVELS = [0.1, 0.5, 1.0, 2.0, 5.0]
    results, clean_results = main(ATTACK_TYPE,TEST_PERTURB_LEVELS)

DFE (Deep Feature Extraction) - Evaluation on Dataset
Method: Triplet Loss + Feature Compression + Template Matching
Architecture: ResNet-like CNN with 13 convolutional layers

EVALUATION METRICS: Accuracy, Precision, Recall, F1-Score
DFE MODEL EVALUATION - PERTURBED DATA
Attack Type: DeepFool
📂 Loading model from Models/DFE_base.pth...
✅ Model loaded: 250 templates, 5 classes

CLEAN DATA EVALUATION

📊 Evaluation Results:
  Accuracy:  0.8990 (89.90%)
  Precision: 0.9000
  Recall:    0.8990
  F1-Score:  0.8983

📂 Loading perturbed data (attack type: DeepFool)...
  Loaded DeepFool eps=0.1: 1000 samples
  Loaded DeepFool eps=0.5: 1000 samples
  Loaded DeepFool eps=1.0: 1000 samples
  Loaded DeepFool eps=2.0: 1000 samples
  Loaded DeepFool eps=5.0: 1000 samples

PERTURBED DATA EVALUATION
  Level 0.1: Acc=0.2180, Precision=0.1067, Recall=0.2180, F1=0.1001
  Level 0.5: Acc=0.2180, Precision=0.1077, Recall=0.2180, F1=0.1002
  Level 1.0: Acc=0.2180, Precision=0.1067, Recall=0.2180, F1=0.1001
 